# JS14 Praktikum 1 - Klasifikasi Biner Sleep Wise

Prediksi apakah seseorang memiliki gangguan tidur berdasarkan dataset Sleep Health.

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

path = Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv')
df = pd.read_csv(path)
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
bp = df['Blood Pressure'].str.split('/', expand=True).astype(float)
df['Systolic BP'], df['Diastolic BP'] = bp[0], bp[1]
df['Has Sleep Disorder'] = (df['Sleep Disorder'] != 'None').astype(int)
df = df.drop(columns=['Person ID', 'Blood Pressure', 'Sleep Disorder'])
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Systolic BP,Diastolic BP,Has Sleep Disorder
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,126.0,83.0,0
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,125.0,80.0,0
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,125.0,80.0,0
3,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,140.0,90.0,1
4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,140.0,90.0,1


In [2]:
X = df.drop(columns='Has Sleep Disorder')
y = df['Has Sleep Disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = [col for col in X.columns if col not in numeric]
preprocessor = ColumnTransformer([('numeric', 'passthrough', numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f'Akurasi: {accuracy_score(y_test, prediction):.2%}')
print(classification_report(y_test, prediction, target_names=['Tidak ada gangguan', 'Ada gangguan']))
print('Confusion matrix:\n', confusion_matrix(y_test, prediction))
with open('sleepwise_binary_model.pkl', 'wb') as file:
    pickle.dump(model, file)

Akurasi: 96.00%
                    precision    recall  f1-score   support

Tidak ada gangguan       0.96      0.98      0.97        44
      Ada gangguan       0.97      0.94      0.95        31

          accuracy                           0.96        75
         macro avg       0.96      0.96      0.96        75
      weighted avg       0.96      0.96      0.96        75

Confusion matrix:
 [[43  1]
 [ 2 29]]
